# s22 — Sub-agents as Tools

**What this teaches:** how to delegate to a focused worker without losing control of the conversation. ADK's `agenttool.New` wraps any `llmagent` as a regular tool — the lead calls it like it would call `bash`, the sub-agent runs its own internal loop, and the result comes back as a tool response. Crucially, control **always returns to the lead** — this is *not* ADK's `transfer_to_agent` handoff.

**Concept anchor:** in Omnis, sub-agents are *tools, not destinations*. The lead orchestrates, sub-agents execute; the topology stays a tree.

## Prerequisites

- A working [GoNB](https://github.com/janpfeifer/gonb) kernel.
- An LLM provider configured via env vars before launching Jupyter, e.g.:
  ```
  export OMNIS_PROVIDER=openai_compat
  export OPENAI_BASE_URL=http://localhost:11434/v1
  export OMNIS_MODEL=qwen2.5-coder
  ```
  Defaults to a local Ollama. Swap in `anthropic` / `openai` / `gemini` for hosted providers — see [docs/providers.md](../../docs/providers.md).

## 1. Imports

We pull in ADK's `llmagent` to construct the sub-agent and `agenttool` to wrap it. The lead is built with `agentkit` as in every other example.

In [ ]:
import (
    "context"
    "fmt"
    "os"

    "google.golang.org/adk/agent/llmagent"
    "google.golang.org/adk/tool"
    "google.golang.org/adk/tool/agenttool"

    "github.com/blouargant/omnis/core/agentkit"
    "github.com/blouargant/omnis/core/stream"
)

## 2. Helper

We swap the example's `os.Exit`-based `must` for a panic version so a notebook error doesn't kill the GoNB kernel.

In [ ]:
func must(err error) {
    if err != nil {
        panic(fmt.Sprintf("%v", err))
    }
}

## 3. Build the sub-agent

The sub-agent is a regular `llmagent` — same model, an explicit `Name`/`Description` that the lead will see in its tool list, and a focused `Instruction`. The description is what the lead's planner reads when deciding whether to call it, so keep it crisp.

In [ ]:
ctx := context.Background()
llm, err := agentkit.NewModel(ctx)
must(err)
sub, err := llmagent.New(llmagent.Config{
    Name:        "summariser",
    Description: "Summarise any text in <=3 sentences.",
    Instruction: "You summarise text in three sentences max.",
    Model:       llm,
})
must(err)

## 4. Wrap as a tool, attach to the lead

`agenttool.New(sub, &agenttool.Config{})` returns a `tool.Tool` that, when invoked, kicks off the sub-agent's own loop and returns its final output. From the lead's perspective this is indistinguishable from any other tool call — but inside, a full nested `model → tool → model` cycle just ran.

In [ ]:
subTool := agenttool.New(sub, &agenttool.Config{})
a, err := agentkit.New(agentkit.AgentConfig{
    Name:        "s22_subagents",
    Description: "Lead agent that calls a summariser sub-agent.",
    Instruction: "When asked to summarise, call the summariser tool with the text.",
    Model:       llm,
    Tools:       []tool.Tool{subTool},
})
must(err)
r, err := agentkit.Runner("s22", a)
must(err)

## 5. Run a prompt that triggers delegation

Ask the lead to summarise a paragraph. You should see exactly one tool call — `summariser(...)` — followed by the lead's own wrap-up sentence acknowledging the delegation.

In [ ]:
prompt := "Summarise this in 3 sentences: Go is a statically typed compiled language. It has goroutines for cheap concurrency. The standard library is famously broad."
must(stream.Print(os.Stdout, agentkit.RunOnce(ctx, r, prompt)))

## What to look for

- After the `summariser` tool call resolves, the **lead** speaks next — never the sub-agent directly. Contrast this with ADK's `transfer_to_agent` flow, which would hand the conversation off and not come back.
- The full Omnis topology (`leader → investigator + summariser + curator`, see [agent/](../../agent/)) is built from this exact primitive, repeated. There is no special "orchestration" code — just nested `agenttool` wrappers.
- Compare with [s26_mailbox](../s26_mailbox/s26_mailbox.ipynb), where agents talk to each other through an FSM mailbox rather than synchronous tool calls.

## Try it yourself

1. Add a second sub-agent called `translator` (instruction: "Translate input text to French.") and ask the lead to summarise *and* translate — observe the lead choosing the order of calls.
2. Set the sub-agent's `Model` to a smaller/cheaper provider while keeping the lead on the strong one — sub-agents are a natural place to use model tiering.